In [ ]:
import pandas as pd
import os 
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path

In [ ]:
filepath = Path("../Dataset/dataset_musique_region.csv")
if not filepath.exists():
    raise FileNotFoundError(f"Fichier introuvable : {filepath.resolve()}. Exécuter d’abord le notebook de nettoyage.")
df = pd.read_csv(filepath)


# Transformation des données

préparer et comparer des variables  pour la prédiction géographique à partir des préférences musicales

- On veut trouver les régions les plus petites



In [24]:
print("Nombre de personnes :", len(df))
print("Nombre de régions :", df["region"].nunique())
print("Région la plus représentée :", df["region"].value_counts().idxmax())
print("Région la moins représentée :", df["region"].value_counts().idxmin())


Nombre de personnes : 1986
Nombre de régions : 12
Région la plus représentée : Île-de-France
Région la moins représentée : Centre-Val de Loire


# A METTRE DANS NETTOYAGE !!!!!!!!!!!!!!!!!!!!!!!!!!

In [6]:
df_counts = df["region"].value_counts().reset_index(name="count")

In [7]:
px.bar(df_counts, x="region", y="count", title="Nombre d'occurences par région")

## Regroupement proposé des régions

- Centre-Val de Loire, Pays de la Loire, Bourgogne-Franche-Comté → **CENTRE**
- Normandie, Bretagne, Hauts-de-France → **NORD-OUEST**
- Provence-Alpes-Côte d’Azur, Occitanie → **SUD**

 Il réduit le nombre de catégories à prédire, mais fait perdre de la précision géographique. 

In [8]:
def groupement(region):
    if region == "Centre-Val de Loire" or region == "Pays de la Loire" or region == "Bourgogne-Franche-Comté":
        return "CENTRE"
    elif region == "Normandie" or region == "Bretagne" or region == "Hauts-de-France":
        return "NORD-OUEST"
    elif region == "Provence-Alpes-Côte d'Azur" or region == "Occitanie":
        return "SUD"
    else:
        return region

In [9]:
df["region_group"] = df["region"].apply(groupement)

In [21]:
df_counts_gr_reg = df["region_group"].value_counts().reset_index(name="count")

In [22]:
px.bar(df_counts_gr_reg, x="region_group", y="count", title="Nombre d'occurences par région")

In [13]:
df["region_group"].value_counts()

region_group
Île-de-France           404
NORD-OUEST              374
SUD                     290
CENTRE                  287
Auvergne-Rhône-Alpes    237
Grand Est               200
Nouvelle-Aquitaine      194
Name: count, dtype: int64

In [28]:
print("Régions originales :", df["region"].nunique())
print("Groupes après transformation :", df["region_group"].nunique())


Régions originales : 12
Groupes après transformation : 7


# Fréquence de chaque style
Chaque colonne musicale vaut 1 si le genre a été sélectionné, 0 sinon. L’objectif est de voir quels genres sont les plus fréquents

In [17]:
for column in df.columns:
    if df[column].dtype == "int64":
        display(df[column].value_counts(normalize=True))
        df_prop = df[column].value_counts(normalize=True).reset_index(name="proportion").sort_values(by=column, ascending=True)
        fig1 = px.bar(df_prop, x=column, y="proportion", title=f"repartiton de {column}")
        fig1.update_layout(yaxis_tickformat=".0%")
        fig1.show()
        fig2 = px.pie(df_prop, names=column, values="proportion", title=f"repartiton de {column}")
        fig2.update_traces(textinfo="percent+label")
        fig2.show()

variete_francaise
1    0.61279
0    0.38721
Name: proportion, dtype: float64

pop_rock
1    0.557402
0    0.442598
Name: proportion, dtype: float64

rap_hip_hop
0    0.719537
1    0.280463
Name: proportion, dtype: float64

classique
0    0.742699
1    0.257301
Name: proportion, dtype: float64

jazz
0    0.808157
1    0.191843
Name: proportion, dtype: float64

...

rnb
0    0.708963
1    0.291037
Name: proportion, dtype: float64

soul
0    0.811682
1    0.188318
Name: proportion, dtype: float64

reggae
0    0.803122
1    0.196878
Name: proportion, dtype: float64

musiques_du_monde
0    0.733132
1    0.266868
Name: proportion, dtype: float64

autres_genres
0    0.926989
1    0.073011
Name: proportion, dtype: float64

# Nombre de styles écoutés

## Comparaison du nombre de genres selon la région

Ca permet de voir si le nombre de genres sélectionnés change d’une région à l’autre. Une différence observée ne prouve pas qu’elle sera utile pour prédire la région.


In [36]:
moyenne_genres = df.groupby("region")["total_genres"].agg(["mean", "count"]).sort_values("mean", ascending=False)
display(moyenne_genres.round(2))
fig = px.bar(moyenne_genres.reset_index(), x="region", y="mean", title="Nombre moyen de genres sélectionnés selon la région", labels={"region": "Région", "mean": "Nombre moyen de genres"})
fig.update_layout(xaxis_tickangle=-45)
fig.show()


,mean,count
region,,
Bourgogne-Franche-Comté,3.67,84
Pays de la Loire,3.64,125
Centre-Val de Loire,3.62,78
Nouvelle-Aquitaine,3.60,194
Grand Est,3.56,200
Occitanie,3.51,161
Provence-Alpes-Côte d'Azur,3.47,129
Normandie,3.42,89
Bretagne,3.38,97


## Vérifications et export

Les 13 genres d’origine et la région initiale restent présents. Les colonnes ajoutées sont `region_group` et `total_genres`. Le fichier nettoyé initial n’est pas écrasé.


In [ ]:
assert "region" in df.columns and "region_group" in df.columns
assert df["region"].notna().all() and df["region_group"].notna().all()
assert len(df) > 0
assert df["total_genres"].eq(df[colonnes_genres].sum(axis=1)).all()

chemin_sortie = Path(filepath).with_name("dataset_musique_region_transforme.csv")

df.to_csv(chemin_sortie, index=False)
print("Fichier transformé enregistré :", chemin_sortie)
print("Dimensions :", df.shape)


Fichier transformé enregistré : ../Dataset/dataset_musique_region_transforme.csv
Dimensions : (1986, 16)


In [38]:
# Bilan
